# Airline Passenger Satisfaction

## 資料讀入
資料位置: /Users/apple/Desktop/Airline Passenger Satisfaction/train.csv

In [80]:
import pandas as pd

In [81]:
df = pd.read_csv("/Users/apple/Desktop/Airline Passenger Satisfaction/train.csv")

## 確認資料欄位＆檢查

In [82]:
import pandas as pd
import numpy as np

## 載入套件

In [83]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier

In [84]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location                      1039

In [85]:
df.describe()

,Unnamed: 0,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
count,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103594.000000
mean,51951.500000,64924.210502,39.379706,1189.448375,2.729683,3.060296,2.756901,2.976883,3.202129,3.250375,3.439396,3.358158,3.382363,3.351055,3.631833,3.304290,3.640428,3.286351,14.815618,15.178678
std,29994.645522,37463.812252,15.114964,997.147281,1.327829,1.525075,1.398929,1.277621,1.329533,1.349509,1.319088,1.332991,1.288354,1.315605,1.180903,1.265396,1.175663,1.312273,38.230901,38.698682
min,0.000000,1.000000,7.000000,31.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,25975.750000,32533.750000,27.000000,414.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,3.000000,3.000000,3.000000,2.000000,0.000000,0.000000
50%,51951.500000,64856.500000,40.000000,843.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,4.000000,4.000000,4.000000,4.000000,4.000000,3.000000,4.000000,3.000000,0.000000,0.000000
75%,77927.250000,97368.250000,51.000000,1743.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,4.000000,4.000000,4.000000,5.000000,4.000000,5.000000,4.000000,12.000000,13.000000
max,103903.000000,129880.000000,85.000000,4983.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,1592.000000,1584.000000


## 資料清洗

### 刪掉無用欄位

In [86]:
drop_cols = ["Unnamed: 0", "id"]
df = df.drop(columns=drop_cols, errors="ignore")

In [87]:
df["satisfaction"] = df["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

### 填補缺失值

In [88]:
if "Arrival Delay in Minutes" in df.columns:
    df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(
        df["Arrival Delay in Minutes"].median()
    )

In [89]:
print("清理後資料形狀:", df.shape)
print(df["satisfaction"].value_counts())

清理後資料形狀: (103904, 23)
satisfaction
0    58879
1    45025
Name: count, dtype: int64


### 預覽資料

In [90]:
print("資料形狀:", df.shape)
print(df.head())
print(df.info())

資料形狀: (103904, 23)
   Gender      Customer Type  Age   Type of Travel     Class  Flight Distance  \
0    Male     Loyal Customer   13  Personal Travel  Eco Plus              460   
1    Male  disloyal Customer   25  Business travel  Business              235   
2  Female     Loyal Customer   26  Business travel  Business             1142   
3  Female     Loyal Customer   25  Business travel  Business              562   
4    Male     Loyal Customer   61  Business travel  Business              214   

   Inflight wifi service  Departure/Arrival time convenient  \
0                      3                                  4   
1                      3                                  2   
2                      2                                  2   
3                      2                                  5   
4                      3                                  3   

   Ease of Online booking  Gate location  ...  Inflight entertainment  \
0                       3              1  

## 分出 X / y

In [91]:
X = df.drop("satisfaction", axis=1)
y = df["satisfaction"]

## 分辨欄位型態

In [92]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("類別欄位:", categorical_features)
print("數值欄位:", numeric_features)

類別欄位: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
數值欄位: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']


## 資料前處理

In [93]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [94]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

In [95]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 切分資料

In [96]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [97]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (83123, 22)
X_test: (20781, 22)
y_train: (83123,)
y_test: (20781,)


## 建立 Random Forest 模型

In [98]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

## 訓練模型

In [99]:
rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Flight Distance',
                                                   'Inflight wifi service',
                                                   'Departure/Arrival time '
                                                   'convenient',
                                                   'Ease of Online booking',
                                                   'Gate location',
                                                   'Food and drink',
                                                   'Online boarding',
                                                   'Seat comfort',
                                                   'Inflight entertainment',
                                                   'On-board service',
                                                   'Leg room s...
                                                   'Checkin service',
                                                   'Inflight service',
                                                   'Cleanliness',
                                                   'Departure Delay in Minutes',
                                                   'Arrival Delay in Minutes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Gender', 'Customer Type',
                                                   'Type of Travel',
                                                   'Class'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                        random_state=42))])

## 預測與評估

In [100]:
y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.964198065540638

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97     11776
           1       0.97      0.95      0.96      9005

    accuracy                           0.96     20781
   macro avg       0.97      0.96      0.96     20781
weighted avg       0.96      0.96      0.96     20781


Confusion Matrix:
[[11519   257]
 [  487  8518]]


## 取出 Feature Importance

In [101]:
feature_names_num = numeric_features

feature_names_cat = rf_model.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .named_steps["onehot"] \
    .get_feature_names_out(categorical_features)

feature_names = list(feature_names_num) + list(feature_names_cat)

rf_importances = rf_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_importances
}).sort_values(by="importance", ascending=False)

print("Top 15 Feature Importance:")
print(importance_df.head(15))

Top 15 Feature Importance:
                           feature  importance
7                  Online boarding    0.166030
2            Inflight wifi service    0.135001
24                  Class_Business    0.076090
23  Type of Travel_Personal Travel    0.063848
9           Inflight entertainment    0.058983
22  Type of Travel_Business travel    0.048678
8                     Seat comfort    0.040230
4           Ease of Online booking    0.038524
25                       Class_Eco    0.035509
11                Leg room service    0.033613
1                  Flight Distance    0.029680
0                              Age    0.028909
20    Customer Type_Loyal Customer    0.026546
12                Baggage handling    0.026135
10                On-board service    0.025056


# 結論：

### 本研究透過 Logistic Regression 分析顧客滿意度的影響因素，結果顯示：

### 旅遊型顧客（Personal Travel）與不忠誠顧客（disloyal customers）顯著較不滿意，而商務旅客則具有較高滿意度。

### 在服務層面：

### 線上登機（Online boarding）、機上 Wi-Fi 與整體服務品質對滿意度具有正向影響。

### 此外，航班延誤（Arrival Delay）對滿意度有明顯負面影響。

# test.csv 測試及資料

## 測試集資料讀入

In [102]:
df_test = pd.read_csv("/Users/apple/Desktop/Airline Passenger Satisfaction/test.csv")

## 資料預處理

In [103]:
df_test = df_test.drop(columns=["Unnamed: 0", "id"], errors="ignore")

df_test["satisfaction"] = df_test["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

if "Arrival Delay in Minutes" in df_test.columns:
    df_test["Arrival Delay in Minutes"] = df_test["Arrival Delay in Minutes"].fillna(
        df_test["Arrival Delay in Minutes"].median()
    )

## 切分資料集

In [104]:
X_test_external = df_test.drop("satisfaction", axis=1)
y_test_external = df_test["satisfaction"]

## 直接套用剛剛訓練的模型 ><

In [105]:
y_pred_external = rf_model.predict(X_test_external)

## 績效評分 Type 1/Type 2 Error

In [106]:
print("External Test Accuracy:", accuracy_score(y_test_external, y_pred_external))
print("\nClassification Report:")
print(classification_report(y_test_external, y_pred_external))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_external, y_pred_external))

External Test Accuracy: 0.9627348321527563

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97     14573
           1       0.97      0.94      0.96     11403

    accuracy                           0.96     25976
   macro avg       0.96      0.96      0.96     25976
weighted avg       0.96      0.96      0.96     25976


Confusion Matrix:
[[14252   321]
 [  647 10756]]
